# GPT-Flux 演示 / Demo

在免费 Colab T4 上运行 GPT-Flux（Qwen3-14B + QLoRA，4-bit 量化后约 9GB 显存）。
Run GPT-Flux on a free Colab T4 GPU (4-bit quantized, ~9GB VRAM).

1. Runtime → Change runtime type → **T4 GPU**
2. 依次运行以下 cells / Run the cells below in order

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.bfloat16)
base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-14B", quantization_config=bnb, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, "LonghaoWang/flux-gpt-qwen3-14b")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-14B", trust_remote_code=True)
print("model ready")

In [ ]:
def ask(q, max_new_tokens=512):
    msgs = [{"role": "system", "content": "你是通量GPT，一位精通地学与通量站观测的中文科研助手。"},
            {"role": "user", "content": q}]
    x = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        y = model.generate(x, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return tok.decode(y[0][x.shape[1]:], skip_special_tokens=True).strip()

print(ask("涡度相关法测量碳通量有哪些主要误差来源？"))

In [ ]:
# 试试更多问题 / Try more questions
for q in ["什么是能量平衡闭合问题？", "FLUXNET 数据在碳循环研究中有什么作用？"]:
    print("问:", q)
    print("答:", ask(q))
    print("-" * 60)